### Dim User ###


In [0]:
import os
import sys
from pyspark.sql.functions import *
os.getcwd()

project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
from utils.transformation import Reusable

#### **Auto Loader** ####


In [0]:
df_user = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimUser/checkpoint/")
    .load("abfss://bronze@myspotifystorage.dfs.core.windows.net/DimUser/")
)
##here we applied the streaming to load the data from the broze to silver blob storage in the azure. 
# spark streaming is used to load the data in the streaming manner when ever the new data is loaded in the bronze blob storage same like incremental loading


In [0]:
df = df_user.withColumn("user_name", upper(col("user_name")))


In [0]:
data_obj = Reusable()
data = data_obj.dropColumns(df,['_rescued_data'])
data = df.dropDuplicates(['user_id'])
display(data, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimUser/checkpoint_dil/")

In [0]:
df_user.writeStream.format("delta")\
        .option("checkpointLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimUser/checkpoint_dile/")\
        .trigger(once=True)\
        .option("path","abfss://silver@myspotifystorage.dfs.core.windows.net/DimUser/data")\
        .toTable("spotify_cata.silver.DimUser")

## DIM Artist


In [0]:
df_artist = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimArtist/checkpoint/")
    .load("abfss://bronze@myspotifystorage.dfs.core.windows.net/DimArtist/")
)

In [0]:
df = df_artist.withColumn("artist_name", upper(col("artist_name")))
#display(df, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimArtist/checkpoint_dil/")

In [0]:
df_art_obj = Reusable()
df_art = df_art_obj.dropColumns(df,['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])
display(df_art, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimArtist/checkpoint_di/")

In [0]:
df_art.writeStream.format("delta")\
        .option("checkpointLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimArtist/checkpoint_dile/")\
        .trigger(once=True)\
        .option("path","abfss://silver@myspotifystorage.dfs.core.windows.net/DimArtist/data")\
        .toTable("spotify_cata.silver.DimArtist")

## Dim Track


In [0]:
df_track = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimTrack/checkpoint/")
    .load("abfss://bronze@myspotifystorage.dfs.core.windows.net/DimTrack/"))

In [0]:
df_track = df_track.withColumn("track_name", upper(col("track_name")))
#display(df, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimTrack/checkpoint_di/")

In [0]:
df_track=df_track.withColumn("duration_flag",when(col("duration_sec") < 150,"Low")\
                            .when(col("duration_sec") > 300,"Medium")\
                            .otherwise("High"))
df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))

display(df_track, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimTrack/checkpoint_dil2/")


In [0]:
df_track_obj = Reusable()
df_track = df_track_obj.dropColumns(df_track,['_rescued_data'])
df_track = df_track.dropDuplicates(['track_id'])


In [0]:
df_track.writeStream.format("delta")\
        .option("checkpointLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimTrack/checkpoint_dill/")\
        .trigger(once=True)\
        .option("path","abfss://silver@myspotifystorage.dfs.core.windows.net/DimTrack/data")\
        .toTable("spotify_cata.silver.DimTrack")

In [0]:
%sql
DROP TABLE spotify_cata.silver.DimArtist;


##Dim Date

In [0]:
df_date = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimDate/checkpoint/")
    .load("abfss://bronze@myspotifystorage.dfs.core.windows.net/DimDate/")
)

In [0]:
display(df, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimDate/checkpoint_dil/")

In [0]:
df_date_obj = Reusable()
df_date = df_date_obj.dropColumns(df_date,['_rescued_data'])
df_date = df_date.dropDuplicates(['date_key'])
display(df_date, checkpointLocation="abfss://silver@myspotifystorage.dfs.core.windows.net/DimDate/checkpoint_di/") 

In [0]:
df_date.writeStream.format("delta")\
        .option("checkpointLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/DimDate/checkpoint_dill/")\
        .trigger(once=True)\
        .option("path","abfss://silver@myspotifystorage.dfs.core.windows.net/DimDate/data")\
        .toTable("spotify_cata.silver.DimDate")

##Fact Stream

In [0]:
df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/FactStream/checkpoint/")
    .load("abfss://bronze@myspotifystorage.dfs.core.windows.net/FactStream/")
)

In [0]:
df_stream_obj = Reusable()
df_stream = df_stream_obj.dropColumns(df_stream,['_rescued_data'])


In [0]:
df_stream.writeStream.format("delta")\
        .option("checkpointLocation", "abfss://silver@myspotifystorage.dfs.core.windows.net/FactStream/checkpoint_dill/")\
        .trigger(once=True)\
        .option("path","abfss://silver@myspotifystorage.dfs.core.windows.net/Factstream/data")\
        .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
drop table if exists spotify_cata.silver.factstream